In [1]:
import os

import numpy as np
import pandas as pd

In [2]:
raw_path = 'data/raw'
processed_path = 'data/processed'

In [3]:
volumetric_df = pd.DataFrame()

# HeatCapacityofDrainedPeatSoils_Gnatowski et al., 2022

In [4]:
def load_f4_frame(file_name, value_column, output_column):
    df = pd.read_csv(f'{raw_path}/{file_name}')
    df.columns = df.columns.str.strip()
    df['Site'] = pd.to_numeric(df['Site'], errors='coerce').astype('Int64')
    df['Soil_Depth_(cm)'] = pd.to_numeric(df['Soil_Depth_(cm)'], errors='coerce')
    df['Number_of_Soil_Sample'] = pd.to_numeric(df['Number_of_Soil_Sample'], errors='coerce')
    df[value_column] = pd.to_numeric(df[value_column], errors='coerce')
    return df[['Site', 'Soil_Depth_(cm)', 'Number_of_Soil_Sample', value_column]].rename(
        columns={
            'Number_of_Soil_Sample': 'Sample_Number',
            value_column: output_column,
        }
    )

def match_f4_frame(base_df, other_df, value_column):
    matched_rows = []
    for (site, depth), base_group in base_df.groupby(['Site', 'Soil_Depth_(cm)'], dropna=False):
        other_group = other_df[
            (other_df['Site'] == site)
            & (other_df['Soil_Depth_(cm)'] == depth)
        ].sort_values('Sample_Number').reset_index(drop=True)
        used = set()
        for _, base_row in base_group.sort_values('Sample_Number').iterrows():
            available = other_group.loc[~other_group.index.isin(used)].copy()
            nearest_idx = (available['Sample_Number'] - base_row['Sample_Number']).abs().idxmin()
            used.add(nearest_idx)
            matched_rows.append({
                'Site': base_row['Site'],
                'Soil_Depth_(cm)': base_row['Soil_Depth_(cm)'],
                'Sample_Number': base_row['Sample_Number'],
                value_column: other_group.loc[nearest_idx, value_column],
            })
    return pd.DataFrame(matched_rows)

f4_volumetric_heat_capacity = load_f4_frame(
    'F4a_Volumetric_Heat_Capacity.csv',
    'Volumetric_Heat_Capacity_(Jcm^-3K^-1)',
    'Volumetric_Heat_Capacity_(Jcm^-3K^-1)'
)

f4_soil_temperature = load_f4_frame(
    'F4b_Soil_Temprerature_vs_Num_Soil_Sample.csv',
    'Soil_Temperature_(C)',
    'Soil_Temperature_(C)'
)

f4_moisture_content = load_f4_frame(
    'F4c_Moisture_Content_vs_Num_Soil_Sample.csv',
    'Moisture_Content_(cm^3cm^-3)',
    'Moisture_Content_(cm^3cm^-3)'
)

f4_bulk_density = load_f4_frame(
    'F4d_Bulk_Density_vs_Num_Soil_Sample.csv',
    'Bulk_Density_(gcm^-3)',
    'Bulk_Density_(gcm^-3)'
)

f4_combined = (
    f4_volumetric_heat_capacity
    .merge(
        match_f4_frame(f4_volumetric_heat_capacity, f4_soil_temperature, 'Soil_Temperature_(C)'),
        on=['Site', 'Soil_Depth_(cm)', 'Sample_Number'],
        how='left'
    )
    .merge(
        match_f4_frame(f4_volumetric_heat_capacity, f4_moisture_content, 'Moisture_Content_(cm^3cm^-3)'),
        on=['Site', 'Soil_Depth_(cm)', 'Sample_Number'],
        how='left'
    )
    .merge(
        match_f4_frame(f4_volumetric_heat_capacity, f4_bulk_density, 'Bulk_Density_(gcm^-3)'),
        on=['Site', 'Soil_Depth_(cm)', 'Sample_Number'],
        how='left'
    )
    .sort_values(['Site', 'Soil_Depth_(cm)', 'Sample_Number'])
    .reset_index(drop=True)
)

output_path = 'data/cleaned/F4_combined_by_soil_sample.csv'
f4_combined.to_csv(output_path, index=False)

In [5]:
f4_combined

,Site,Soil_Depth_(cm),Sample_Number,Volumetric_Heat_Capacity_(Jcm^-3K^-1),Soil_Temperature_(C),Moisture_Content_(cm^3cm^-3),Bulk_Density_(gcm^-3)
0,1,5.0,0.88102,3.28469,22.76121,0.64677,0.31991
1,1,5.0,1.94194,3.28848,22.80974,0.64677,0.30815
2,1,5.0,2.95726,3.16926,22.47323,0.67043,0.28780
3,1,7.5,3.96970,3.18440,22.54575,0.69804,0.35139
4,1,7.5,4.92783,3.48339,22.42584,0.72071,0.32627
...,...,...,...,...,...,...,...
61,5,12.5,61.94734,3.23558,11.28207,0.65959,0.22104
62,5,12.5,62.99941,3.31837,11.93205,0.71085,0.24711
63,5,22.5,63.95662,3.48334,12.31736,0.77617,0.22740
64,5,22.5,64.91286,3.64344,11.95686,0.80944,0.23694


In [6]:
new_rows = f4_combined[[
    'Site',
    'Soil_Depth_(cm)',
    'Moisture_Content_(cm^3cm^-3)',
    'Volumetric_Heat_Capacity_(Jcm^-3K^-1)'
]].rename(columns={
    'Moisture_Content_(cm^3cm^-3)': 'Water_Content',
    'Volumetric_Heat_Capacity_(Jcm^-3K^-1)': 'Volumetric_Heat_Capacity'
})

new_rows['Bulk_Density'] = np.nan
new_rows['Data_Type'] = 'Observed'
new_rows['Soil'] = 'Peat'
new_rows['Source'] = 'Gnatowski et al. 2022 - F4'
new_rows['Water_Content_Basis'] = 'volumetric'
new_rows = new_rows[['Site', 'Soil_Depth_(cm)', 'Water_Content', 'Water_Content_Basis', 'Volumetric_Heat_Capacity', 'Bulk_Density', 'Data_Type', 'Soil', 'Source']]

volumetric_df = pd.concat([volumetric_df, new_rows], ignore_index=True)

In [7]:
volumetric_df.tail()

,Site,Soil_Depth_(cm),Water_Content,Water_Content_Basis,Volumetric_Heat_Capacity,Bulk_Density,Data_Type,Soil,Source
61,5,12.5,0.65959,volumetric,3.23558,NaN,Observed,Peat,Gnatowski et al. 2022 - F4
62,5,12.5,0.71085,volumetric,3.31837,NaN,Observed,Peat,Gnatowski et al. 2022 - F4
63,5,22.5,0.77617,volumetric,3.48334,NaN,Observed,Peat,Gnatowski et al. 2022 - F4
64,5,22.5,0.80944,volumetric,3.64344,NaN,Observed,Peat,Gnatowski et al. 2022 - F4
65,5,22.5,0.78110,volumetric,3.58378,NaN,Observed,Peat,Gnatowski et al. 2022 - F4


In [8]:
df_3 = pd.read_csv(f'{raw_path}/F7a_CV_vs_Soil_Moisure_Content.csv')
df_3.columns = df_3.columns.str.strip()

In [9]:
new_rows = df_3[[
    'Site',
    'Soil_Moisture_Content_(cm^3cm^-3)',
    'Cv_(Jcm^-3K^-1)'
]].rename(columns={
    'Soil_Moisture_Content_(cm^3cm^-3)': 'Water_Content',
    'Cv_(Jcm^-3K^-1)': 'Volumetric_Heat_Capacity'
})

new_rows['Soil_Depth_(cm)'] = np.nan
new_rows['Bulk_Density'] = np.nan
new_rows['Data_Type'] = 'Observed'
new_rows['Soil'] = 'Peat'
new_rows['Source'] = 'Gnatowski et al. 2022 - F7a'
new_rows['Water_Content_Basis'] = 'volumetric'
new_rows = new_rows[['Site', 'Soil_Depth_(cm)', 'Water_Content', 'Water_Content_Basis', 'Volumetric_Heat_Capacity', 'Bulk_Density', 'Data_Type', 'Soil', 'Source']]

volumetric_df = pd.concat([volumetric_df, new_rows], ignore_index=True)
volumetric_df.tail()

,Site,Soil_Depth_(cm),Water_Content,Water_Content_Basis,Volumetric_Heat_Capacity,Bulk_Density,Data_Type,Soil,Source
82,4,NaN,0.90474,volumetric,3.75439,NaN,Observed,Peat,Gnatowski et al. 2022 - F7a
83,5,NaN,0.92337,volumetric,3.28899,NaN,Observed,Peat,Gnatowski et al. 2022 - F7a
84,5,NaN,0.91799,volumetric,3.30933,NaN,Observed,Peat,Gnatowski et al. 2022 - F7a
85,5,NaN,0.91235,volumetric,3.55236,NaN,Observed,Peat,Gnatowski et al. 2022 - F7a
86,5,NaN,0.92963,volumetric,2.83352,NaN,Observed,Peat,Gnatowski et al. 2022 - F7a


# TheramlProperties_Nidal H. Abu-Hamdeh_2003

In [10]:
df_8 = pd.read_csv(f'{raw_path}/F3_Clay_Volumetric_Heat_Capacity.csv')
df_8.columns = df_8.columns.str.strip()
df_8['Moisture_Content_w_(kg/kg)'] = pd.to_numeric(df_8['Moisture_Content_w_(kg/kg)'], errors='coerce')
df_8['Volumetric_Heat_Capacity_Cv_(MJ/m^3)'] = pd.to_numeric(df_8['Volumetric_Heat_Capacity_Cv_(MJ/m^3)'], errors='coerce')
df_8['Bulk_Density_(kgm^-3)'] = pd.to_numeric(df_8['Bulk_Density_(kgm^-3)'], errors='coerce')
df_8['Data_Type'] = df_8['Data_Type'].str.strip()

# 1 MJ/m^3/K is numerically equal to 1 J/cm^3/K.
df_8['Volumetric_Heat_Capacity_(Jcm^-3K^-1)'] = df_8['Volumetric_Heat_Capacity_Cv_(MJ/m^3)']

new_rows = df_8.rename(columns={
    'Moisture_Content_w_(kg/kg)': 'Water_Content',
    'Volumetric_Heat_Capacity_(Jcm^-3K^-1)': 'Volumetric_Heat_Capacity',
    'Bulk_Density_(kgm^-3)': 'Bulk_Density'
})
new_rows['Soil'] = 'Clay'
new_rows['Water_Content_Basis'] = 'gravimetric'
new_rows['Source'] = 'Abu-Hamdeh 2003'
new_rows['Soil_Depth_(cm)'] = np.nan
new_rows['Site'] = np.nan
new_rows = new_rows[['Site', 'Soil_Depth_(cm)', 'Water_Content', 'Water_Content_Basis', 'Volumetric_Heat_Capacity', 'Bulk_Density', 'Data_Type', 'Soil', 'Source']]

volumetric_df = pd.concat([volumetric_df, new_rows], ignore_index=True)
volumetric_df.tail()

,Site,Soil_Depth_(cm),Water_Content,Water_Content_Basis,Volumetric_Heat_Capacity,Bulk_Density,Data_Type,Soil,Source
112,<NA>,NaN,-0.00017,gravimetric,1.41550,1400.0,Predicted,Clay,Abu-Hamdeh 2003
113,<NA>,NaN,0.06006,gravimetric,1.75949,1400.0,Predicted,Clay,Abu-Hamdeh 2003
114,<NA>,NaN,0.12987,gravimetric,2.17225,1400.0,Predicted,Clay,Abu-Hamdeh 2003
115,<NA>,NaN,0.20006,gravimetric,2.59263,1400.0,Predicted,Clay,Abu-Hamdeh 2003
116,<NA>,NaN,0.25031,gravimetric,2.88311,1400.0,Predicted,Clay,Abu-Hamdeh 2003


In [11]:
df_9 = pd.read_csv(f'{raw_path}/F4_Sandy_Soil_Volumetric_Heat_Capacity.csv')
df_9.columns = df_9.columns.str.strip()
df_9['Moisture_Content_w_(kg/kg)'] = pd.to_numeric(df_9['Moisture_Content_w_(kg/kg)'], errors='coerce')
df_9['Volumetric_Heat_Capacity_Cv_(MJ/m^3)'] = pd.to_numeric(df_9['Volumetric_Heat_Capacity_Cv_(MJ/m^3)'], errors='coerce')
df_9['Bulk_Density_(kgm^-3)'] = pd.to_numeric(df_9['Bulk_Density_(kgm^-3)'], errors='coerce')
df_9['Data_Type'] = df_9['Data_Type'].str.strip()

# 1 MJ/m^3/K is numerically equal to 1 J/cm^3/K.
df_9['Volumetric_Heat_Capacity_(Jcm^-3K^-1)'] = df_9['Volumetric_Heat_Capacity_Cv_(MJ/m^3)']

new_rows = df_9.rename(columns={
    'Moisture_Content_w_(kg/kg)': 'Water_Content',
    'Volumetric_Heat_Capacity_(Jcm^-3K^-1)': 'Volumetric_Heat_Capacity',
    'Bulk_Density_(kgm^-3)': 'Bulk_Density'
})
new_rows['Soil'] = 'Sand'
new_rows['Water_Content_Basis'] = 'gravimetric'
new_rows['Source'] = 'Abu-Hamdeh 2003'
new_rows['Soil_Depth_(cm)'] = np.nan
new_rows['Site'] = np.nan
new_rows = new_rows[['Site', 'Soil_Depth_(cm)', 'Water_Content', 'Water_Content_Basis', 'Volumetric_Heat_Capacity', 'Bulk_Density', 'Data_Type', 'Soil', 'Source']]

volumetric_df = pd.concat([volumetric_df, new_rows], ignore_index=True)
volumetric_df.tail()

,Site,Soil_Depth_(cm),Water_Content,Water_Content_Basis,Volumetric_Heat_Capacity,Bulk_Density,Data_Type,Soil,Source
142,<NA>,NaN,0.00000,gravimetric,1.00318,1400.0,Predicted,Sand,Abu-Hamdeh 2003
143,<NA>,NaN,0.05978,gravimetric,1.34593,1400.0,Predicted,Sand,Abu-Hamdeh 2003
144,<NA>,NaN,0.13007,gravimetric,1.74580,1400.0,Predicted,Sand,Abu-Hamdeh 2003
145,<NA>,NaN,0.20000,gravimetric,2.14567,1400.0,Predicted,Sand,Abu-Hamdeh 2003
146,<NA>,NaN,0.25000,gravimetric,2.39949,1400.0,Predicted,Sand,Abu-Hamdeh 2003


## Standardize For Modeling

In [12]:
volumetric_df = volumetric_df.copy()
for column in ['Site', 'Soil_Depth_(cm)', 'Water_Content', 'Water_Content_Basis', 'Volumetric_Heat_Capacity', 'Bulk_Density', 'Data_Type', 'Soil', 'Source']:
    if column not in volumetric_df.columns:
        volumetric_df[column] = np.nan

volumetric_df['Data_Type'] = volumetric_df['Data_Type'].fillna('Observed')
volumetric_df['Soil'] = volumetric_df['Soil'].fillna('Unknown')

cv_model_df = volumetric_df.rename(columns={
    'Volumetric_Heat_Capacity': 'target_value',
    'Water_Content': 'water_content',
    'Water_Content_Basis': 'water_content_basis',
    'Bulk_Density': 'bulk_density',
    'Data_Type': 'data_type',
    'Soil': 'soil_class',
    'Source': 'source'
}).copy()
cv_model_df['temperature_c'] = np.nan
cv_model_df['dataset_family'] = 'volumetric'
cv_model_df['target_type'] = 'volumetric_heat_capacity'
cv_model_df = cv_model_df[[
    'target_value', 'target_type', 'dataset_family',
    'water_content', 'water_content_basis', 'temperature_c', 'bulk_density',
    'soil_class', 'data_type', 'source', 'Site', 'Soil_Depth_(cm)'
]]

cv_model_df.head()

,target_value,target_type,dataset_family,water_content,water_content_basis,temperature_c,bulk_density,soil_class,data_type,source,Site,Soil_Depth_(cm)
0,3.28469,volumetric_heat_capacity,volumetric,0.64677,volumetric,NaN,NaN,Peat,Observed,Gnatowski et al. 2022 - F4,1.0,5.0
1,3.28848,volumetric_heat_capacity,volumetric,0.64677,volumetric,NaN,NaN,Peat,Observed,Gnatowski et al. 2022 - F4,1.0,5.0
2,3.16926,volumetric_heat_capacity,volumetric,0.67043,volumetric,NaN,NaN,Peat,Observed,Gnatowski et al. 2022 - F4,1.0,5.0
3,3.18440,volumetric_heat_capacity,volumetric,0.69804,volumetric,NaN,NaN,Peat,Observed,Gnatowski et al. 2022 - F4,1.0,7.5
4,3.48339,volumetric_heat_capacity,volumetric,0.72071,volumetric,NaN,NaN,Peat,Observed,Gnatowski et al. 2022 - F4,1.0,7.5


## Optional Specific-Heat Candidates From Cv

In [13]:
cp_from_cv_candidates_df = volumetric_df[
    volumetric_df['Bulk_Density'].notna()
].copy()

# Here Bulk_Density is in kg/m^3 and Volumetric_Heat_Capacity is numerically equal to MJ/m^3/K.
cp_from_cv_candidates_df['target_value'] = (
    1_000_000 * cp_from_cv_candidates_df['Volumetric_Heat_Capacity']
    / cp_from_cv_candidates_df['Bulk_Density']
)
cp_from_cv_candidates_df = cp_from_cv_candidates_df.rename(columns={
    'Water_Content': 'water_content',
    'Water_Content_Basis': 'water_content_basis',
    'Bulk_Density': 'bulk_density',
    'Data_Type': 'data_type',
    'Soil': 'soil_class',
    'Source': 'source'
})
cp_from_cv_candidates_df['temperature_c'] = np.nan
cp_from_cv_candidates_df['dataset_family'] = 'converted_from_cv'
cp_from_cv_candidates_df['target_type'] = 'specific_heat'
cp_from_cv_candidates_df = cp_from_cv_candidates_df[[
    'target_value', 'target_type', 'dataset_family',
    'water_content', 'water_content_basis', 'temperature_c', 'bulk_density',
    'soil_class', 'data_type', 'source'
]]

cp_from_cv_candidates_df.head()

,target_value,target_type,dataset_family,water_content,water_content_basis,temperature_c,bulk_density,soil_class,data_type,source
87,1370.183333,specific_heat,converted_from_cv,0.00004,gravimetric,NaN,1200.0,Clay,Observed,Abu-Hamdeh 2003
88,1704.491667,specific_heat,converted_from_cv,0.05988,gravimetric,NaN,1200.0,Clay,Observed,Abu-Hamdeh 2003
89,2083.391667,specific_heat,converted_from_cv,0.12989,gravimetric,NaN,1200.0,Clay,Observed,Abu-Hamdeh 2003
90,2468.658333,specific_heat,converted_from_cv,0.20009,gravimetric,NaN,1200.0,Clay,Observed,Abu-Hamdeh 2003
91,2783.783333,specific_heat,converted_from_cv,0.25015,gravimetric,NaN,1200.0,Clay,Observed,Abu-Hamdeh 2003


In [14]:
os.makedirs(processed_path, exist_ok=True)
volumetric_df.to_csv(f'{processed_path}/volumetric_cleaned.csv', index=False)
cv_model_df.to_csv(f'{processed_path}/cv_model_df.csv', index=False)
cp_from_cv_candidates_df.to_csv(
    f'{processed_path}/cp_moisture_density_candidates_df.csv',
    index=False,
)